## Prepare the dataset for pretraining

In [3]:
from datasets import load_dataset
from datasets import load_dataset, DatasetDict

c:\Users\gaura\Downloads\llmwithrope\llmrope\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
ds = load_dataset("Salesforce/wikitext", "wikitext-103-v1")

Generating validation split: 100%|██████████| 3760/3760 [00:00<00:00, 79928.35 examples/s]


In [16]:
train_subset = ds["train"].shuffle(seed=42).select(range(50000))

# Shuffle and pick 3k rows from test data
test_subset = ds["test"].shuffle(seed=42).select(range(3000))

# Combine into a DatasetDict
subset_ds = DatasetDict({
    "train": train_subset,
    "test": test_subset
})

# Save to a folder named 'dataset'
subset_ds.save_to_disk("dataset")


Saving the dataset (1/1 shards): 100%|██████████| 3000/3000 [00:00<00:00, 172989.52 examples/s]


In [4]:
import tiktoken

class Tokenizer:
    def __init__(self, encoding_name="gpt2"):
        # Load a tiktoken encoding (GPT-2 vocab is common)
        self.enc = tiktoken.get_encoding(encoding_name)

        # Vocabulary size
        self.vocab_size = self.enc.n_vocab

    # Convert text string to list of token IDs
    def encode(self, text):
        return self.enc.encode(text)

    # Convert list of token IDs back to text string
    def decode(self, ids):
        return self.enc.decode(ids)

# Example usage
training_text = "The cat sat on the mat."
tokenizer = Tokenizer("gpt2")

tokens = tokenizer.encode(training_text)
print("Tokens:", tokens)

decoded = tokenizer.decode(tokens)
print("Decoded:", decoded)

print(f"Vocabulary size: {tokenizer.vocab_size}")

Tokens: [464, 3797, 3332, 319, 262, 2603, 13]
Decoded: The cat sat on the mat.
Vocabulary size: 50257


In [5]:
from datasets import load_from_disk

# Load the dataset from the folder
subset_ds = load_from_disk("dataset")

# Inspect the splits
print(subset_ds)

# Peek at a few rows from train
print(subset_ds["train"][0:5])


DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 50000
    })
    test: Dataset({
        features: ['text'],
        num_rows: 3000
    })
})
{'text': [' = = = Middle East = = = \n', '', ' = = = Rider breakdown = = = \n', '', ' The Mesozoic era is represented in the park by the model rock exposure showing a succession of beds , namely the Jurassic and Cretaceous , by models of dinosaurs and other animals known from <unk> fossils , and by suitable vegetation - both living plants and models . \n']}


In [6]:
import random

texts = [row["text"].strip() for row in subset_ds["train"] if row["text"].strip()]
random.shuffle(texts)
training_text = "\n\n".join(texts)

In [7]:
tokenizer = Tokenizer("gpt2")

tokens = tokenizer.encode(training_text)
print("Tokens:", tokens[:20])


Tokens: [28, 796, 796, 1605, 5050, 290, 10064, 796, 796, 796, 198, 198, 38, 8126, 2722, 27309, 3562, 416, 402, 1504]


In [8]:
from torch.utils.data import Dataset, DataLoader

# Dataset for language modeling
class TextDataset(Dataset):
  def __init__(self, text, tokenizer, max_seq_length):
    # Convert text to token IDs
    self.tokens = tokenizer.encode(text)

    # Maximum length of each training sequence
    self.max_seq_length = max_seq_length

    # Number of valid training sequences
    self.num_sequences = (len(self.tokens) - 1) // max_seq_length

  # Total number of valid training sequences
  def __len__(self):
    return self.num_sequences
  
  # Get an input sequence and targets
  def __getitem__(self, idx):
    # Start index of the sequence
    start = idx * self.max_seq_length

    # End index of the sequence
    end = start + self.max_seq_length

    # Input token sequence
    input_ids = torch.tensor(self.tokens[start:end], dtype=torch.long)

    # Next-token targets/ labels (shifted by one character)
    target_ids = torch.tensor(self.tokens[start+1:end+1], dtype=torch.long)

    return input_ids, target_ids

In [9]:
# Define maximum training sequence length
max_seq_length = 512

# Create dataset
train_dataset = TextDataset(training_text, tokenizer, max_seq_length)

In [10]:
print(f"Number of training sequences: {len(train_dataset):,}")

# Output: Number of training sequences: 4,186,898

Number of training sequences: 6,359


In [11]:
input, target = train_dataset[10]

print("Input IDs:\n", input)
print("\nTarget IDs:\n", target)

print("\nDecoded Input:\n", tokenizer.decode(input.tolist()))
print("\nDecoded Target:\n", tokenizer.decode(target.tolist()))

Input IDs:
 tensor([ 4816,   318,   257,  7169,    82,  2223,  8855,  3195,  2168,   764,
         4100,  2539,   318,  2077,   284, 23668,   284,   262, 29504,  8092,
        26690,   837,   543,   318,  1912,   319,   257,  1103,  2488,    12,
           31,  1204,  4436,   764,   198,   198,   818,  8309,   837,   262,
        30183,  5209, 19954,   286, 14015, 18322, 11294, 23117,  1192,   292,
          837, 32705,  8047,  3254,   641,    72,   837, 12702,   396, 48127,
         1872,  1279,  2954,    29,   837,   290, 34269, 14236, 47847,   952,
         3125, 35671,   764, 11294, 23117,  1192,   292,   705,    82,  2239,
        11358,   290,  3125, 35671,   705,    82,   290,  1279,  2954,    29,
          705,    82,  4697,  9397,  5495,   262, 28176,   316,   284,   262,
         2647,   286,   842, 25002,  6802,  5811,  1526,  1636,   837,  1237,
          404,  2954,  1448,   262, 36271, 22153,   837,   290,  5559,  3881,
         4097, 12091,   705,    82, 40187,   764, 36

In [12]:
# Define batch size
batch_size = 16

# Create training DataLoader
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)

In [13]:
print(f"Number of training sequences: {len(train_dataset):,}")

print(f"Number of batches per epoch: {len(train_loader):,}") # Number of training sequences / batch_size

Number of training sequences: 6,359
Number of batches per epoch: 398


In [14]:
import torch
import torch.nn as nn
import math

def build_rope_cache(max_seq_len, head_dim, device, base=10000):
    """
    Precompute cos/sin rotation frequencies for RoPE.
    Returns tensors of shape [max_seq_len, head_dim].
    """
    assert head_dim % 2 == 0, "head_dim must be even for RoPE"

    half_dim = head_dim // 2
    dim_indices = torch.arange(half_dim, device=device)
    inv_freq = 1.0 / (base ** (dim_indices / half_dim))

    positions = torch.arange(max_seq_len, device=device)
    angle_matrix = torch.outer(positions, inv_freq)  # [seq_len, half_dim]

    cos_cache = torch.cos(angle_matrix)
    sin_cache = torch.sin(angle_matrix)

    # Duplicate to match full head_dim
    cos_cache = torch.cat([cos_cache, cos_cache], dim=-1)  # [seq_len, head_dim]
    sin_cache = torch.cat([sin_cache, sin_cache], dim=-1)

    return cos_cache, sin_cache


def apply_rope(q_or_k, cos_cache, sin_cache):
    """
    Apply RoPE rotation to queries or keys.
    q_or_k: [batch, heads, seq_len, head_dim]
    cos/sin: [seq_len, head_dim]
    """
    batch_size, num_heads, seq_len, head_dim = q_or_k.shape
    half_dim = head_dim // 2

    # Split into two halves
    first_half = q_or_k[..., :half_dim]
    second_half = q_or_k[..., half_dim:]

    # Broadcast cos/sin to match [batch, heads, seq_len, half_dim]
    cos = cos_cache[:seq_len, :half_dim].unsqueeze(0).unsqueeze(0)
    sin = sin_cache[:seq_len, :half_dim].unsqueeze(0).unsqueeze(0)

    rotated_first = first_half * cos - second_half * sin
    rotated_second = first_half * sin + second_half * cos

    return torch.cat([rotated_first, rotated_second], dim=-1)


class CausalMultiHeadSelfAttention(nn.Module):
    def __init__(self, embed_dim, num_heads, rope_base=10000):
        super().__init__()
        assert embed_dim % num_heads == 0, "embed_dim must be divisible by num_heads"
        assert (embed_dim // num_heads) % 2 == 0, "head_dim must be even for RoPE"

        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.rope_base = rope_base

        # Linear projections
        self.query_proj = nn.Linear(embed_dim, embed_dim, bias=False)
        self.key_proj = nn.Linear(embed_dim, embed_dim, bias=False)
        self.value_proj = nn.Linear(embed_dim, embed_dim, bias=False)
        self.out_proj = nn.Linear(embed_dim, embed_dim, bias=False)

        # Cache for RoPE
        self.register_buffer("cos_cache", None, persistent=False)
        self.register_buffer("sin_cache", None, persistent=False)
        self.max_seq_len_cached = 0

    def _split_heads(self, x):
        batch_size, seq_len, embed_dim = x.shape
        x = x.view(batch_size, seq_len, self.num_heads, self.head_dim)
        return x.transpose(1, 2)  # [batch, heads, seq_len, head_dim]

    def _merge_heads(self, x):
        batch_size, num_heads, seq_len, head_dim = x.shape
        x = x.transpose(1, 2).contiguous()
        return x.view(batch_size, seq_len, self.embed_dim)

    def _update_rope_cache(self, seq_len, device):
        if self.cos_cache is None or seq_len > self.max_seq_len_cached or self.cos_cache.device != device:
            self.cos_cache, self.sin_cache = build_rope_cache(seq_len, self.head_dim, device, self.rope_base)
            self.max_seq_len_cached = seq_len

    def forward(self, x):
        batch_size, seq_len, embed_dim = x.shape

        # Project to Q, K, V
        queries = self.query_proj(x)
        keys = self.key_proj(x)
        values = self.value_proj(x)

        # Split into heads
        q_heads = self._split_heads(queries)
        k_heads = self._split_heads(keys)
        v_heads = self._split_heads(values)

        # Update RoPE cache
        self._update_rope_cache(seq_len, x.device)

        # Apply RoPE to Q and K
        q_heads = apply_rope(q_heads, self.cos_cache, self.sin_cache)
        k_heads = apply_rope(k_heads, self.cos_cache, self.sin_cache)

        # Scaled dot-product attention
        attn_scores = torch.matmul(q_heads, k_heads.transpose(-2, -1)) / math.sqrt(self.head_dim)

        # Causal mask
        causal_mask = torch.tril(torch.ones(seq_len, seq_len, device=x.device))
        attn_scores = attn_scores.masked_fill(causal_mask == 0, float('-inf'))

        attn_weights = torch.softmax(attn_scores, dim=-1)
        context = torch.matmul(attn_weights, v_heads)

        # Merge heads and project output
        merged = self._merge_heads(context)
        return self.out_proj(merged)

In [15]:
# Feed forward network block

class FeedForwardNetwork(nn.Module):
  def __init__(self, embedding_dim, ff_dim, dropout = 0.1):
    super().__init__()

    self.fc1 = nn.Linear(embedding_dim, ff_dim) # Expand feature space to ff_dim (dimension of FFN)

    self.activation = nn.GELU() # Introduce non-linearity

    self.fc2 = nn.Linear(ff_dim, embedding_dim) # Project back to embedding_dim (original embedding dimension)

    self.dropout = nn.Dropout(dropout) # Regularization with Dropout

  def forward(self, x):
    x = self.fc1(x)
    x = self.activation(x)
    x = self.dropout(x)
    x = self.fc2(x)

    return x

In [16]:
# Decoder block
class Decoder(nn.Module):
    def __init__(self, embedding_dim, ff_dim, num_heads, dropout=0.1):
        super().__init__()

        self.attention = CausalMultiHeadSelfAttention(embedding_dim,
                         num_heads)

        self.ffn = FeedForwardNetwork(embedding_dim, ff_dim, dropout)

        # LayerNorm is applied before each sublayer (Pre-LN)
        self.ln1 = nn.LayerNorm(embedding_dim)
        self.ln2 = nn.LayerNorm(embedding_dim)

        # Dropout applied to sublayer outputs for regularization
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # Causal MHA
        attention_output = self.attention(self.ln1(x))

        # Residual connection
        x = x + self.dropout(attention_output)

        # Feed-forward network
        ffn_output = self.ffn(self.ln2(x))

        # Residual connection
        x = x + self.dropout(ffn_output)

        return x

In [17]:
import torch
import torch.nn as nn

class DecoderOnlyTransformer(nn.Module):
    def __init__(self, vocab_size, embedding_dim, num_heads, ff_dim,
                 num_layers, max_seq_length, dropout=0.1):
        super().__init__()

        self.embedding_dim = embedding_dim
        self.max_seq_length = max_seq_length

        # Token embeddings only (no positional embedding needed with RoPE)
        self.token_embedding = nn.Embedding(vocab_size, embedding_dim)

        # Dropout
        self.dropout = nn.Dropout(dropout)

        # Stack of Decoder blocks (these should internally use your RoPE attention)
        self.decoders = nn.ModuleList([
            Decoder(embedding_dim, ff_dim, num_heads, dropout)
            for _ in range(num_layers)
        ])

        # LayerNorm
        self.final_ln = nn.LayerNorm(embedding_dim)

        # Output projection to vocabulary
        self.output_proj = nn.Linear(embedding_dim, vocab_size)

    def forward(self, x):
        # x represents token indices
        batch_size, seq_length = x.shape

        # Token embedding only
        token_embedding = self.token_embedding(x)

        # Apply dropout
        x = self.dropout(token_embedding)

        # Forward pass through decoder blocks
        for decoder in self.decoders:
            x = decoder(x)

        # Apply LayerNorm to the output
        x = self.final_ln(x)

        # Output projection to vocabulary to get logits
        logits = self.output_proj(x)

        return logits

In [18]:
# Create an instance of tokenizer
tokenizer = Tokenizer("gpt2")

# Vocabulary size 
vocab_size = tokenizer.vocab_size
print(f"Vocabulary size: {vocab_size}")

Vocabulary size: 50257


In [19]:
# Model hyperparameters
embedding_dim = 256
ff_dim = 1024  # 4x embedding_dim
num_heads = 8
num_layers = 6
dropout = 0.1

# Initialize model and move it to CPU/GPU
model = DecoderOnlyTransformer(
  vocab_size = vocab_size,
  embedding_dim = embedding_dim,
  num_heads = num_heads,
  ff_dim = ff_dim,
  num_layers = num_layers,
  max_seq_length = max_seq_length,
  dropout = dropout
).to(device)

In [20]:
# Check model parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())

print(f"Trainable parameters: {trainable_params:,} ({trainable_params/1e6:.2f}M)")
print(f"Total parameters: {total_params:,} ({total_params/1e6:.2f}M)\n")

Trainable parameters: 30,514,769 (30.51M)
Total parameters: 30,514,769 (30.51M)



In [21]:
print(model)

DecoderOnlyTransformer(
  (token_embedding): Embedding(50257, 256)
  (dropout): Dropout(p=0.1, inplace=False)
  (decoders): ModuleList(
    (0-5): 6 x Decoder(
      (attention): CausalMultiHeadSelfAttention(
        (query_proj): Linear(in_features=256, out_features=256, bias=False)
        (key_proj): Linear(in_features=256, out_features=256, bias=False)
        (value_proj): Linear(in_features=256, out_features=256, bias=False)
        (out_proj): Linear(in_features=256, out_features=256, bias=False)
      )
      (ffn): FeedForwardNetwork(
        (fc1): Linear(in_features=256, out_features=1024, bias=True)
        (activation): GELU(approximate='none')
        (fc2): Linear(in_features=1024, out_features=256, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (ln1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (ln2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
  )
  (final_ln): Lay

In [22]:
from torch.nn.utils import clip_grad_norm_
from tqdm import tqdm

# Training function for one epoch
def train(model, train_loader, optimizer, criterion, device, epoch):
  # Enable training mode
  model.train()

  # Track total loss
  total_loss = 0.0

  # Total number of batches
  num_batches = len(train_loader)

  for batch_idx, (input_ids, target_ids) in enumerate(tqdm(train_loader, desc = f"Epoch {epoch}", leave = False)):
    # Move batch to CPU or GPU
    input_ids = input_ids.to(device)
    target_ids = target_ids.to(device)

    # Forward pass
    logits = model(input_ids)

    # Calculate loss
    loss = criterion(
            logits.reshape(-1, logits.size(-1)),  # (batch_size * sequence_length, vocab_size)
            target_ids.reshape(-1) # (batch_size * sequence_length)
           )

    # Backward pass
    # Clear old gradients
    optimizer.zero_grad()

    # Compute new gradients
    loss.backward()

    # Clip the gradient norm to prevent exploding gradients
    clip_grad_norm_(model.parameters(), max_norm = 1.0)

    # Update model weights
    optimizer.step()

    # Accumulate loss
    total_loss += loss.item()

  # Return epoch average loss
  return total_loss / num_batches

In [23]:
import torch.optim as optim

# Learning rate
learning_rate = 3e-4

# Optimizer
optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=0.01)

# Loss function
criterion = nn.CrossEntropyLoss()

# Total epochs to train the model for
total_epochs = 3

In [24]:
# LR scheduler (smoothly decreases the learning rate following a cosine curve over training)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_epochs)


# Training for 'total_epochs'
for epoch in range(1, total_epochs + 1):
  avg_loss_per_epoch = train(model, train_loader, optimizer, criterion, device, epoch)

  scheduler.step()

  print(f"Epoch {epoch}/{total_epochs} Complete | Average Loss Per Epoch: {avg_loss_per_epoch:.4f}")

print("Training complete!")

Epoch 1/3 Complete | Average Loss Per Epoch: 6.7907


Epoch 2/3 Complete | Average Loss Per Epoch: 5.9408


Epoch 3/3 Complete | Average Loss Per Epoch: 5.6972
Training complete!


In [30]:
# After training - DIRECTLY save entire model
torch.save(model, 'rope_attention_model.pt')



In [32]:
# Load anywhere
model = torch.load('rope_attention_model.pt', weights_only=False)
model.eval()

DecoderOnlyTransformer(
  (token_embedding): Embedding(50257, 256)
  (dropout): Dropout(p=0.1, inplace=False)
  (decoders): ModuleList(
    (0-5): 6 x Decoder(
      (attention): CausalMultiHeadSelfAttention(
        (query_proj): Linear(in_features=256, out_features=256, bias=False)
        (key_proj): Linear(in_features=256, out_features=256, bias=False)
        (value_proj): Linear(in_features=256, out_features=256, bias=False)
        (out_proj): Linear(in_features=256, out_features=256, bias=False)
      )
      (ffn): FeedForwardNetwork(
        (fc1): Linear(in_features=256, out_features=1024, bias=True)
        (activation): GELU(approximate='none')
        (fc2): Linear(in_features=1024, out_features=256, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (ln1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (ln2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
  )
  (final_ln): Lay

In [33]:
# Function to generate text with Greedy decoding
def generate_text(model, tokenizer, prompt, max_new_tokens, device):

    # Enable evaluation mode
    model.eval()

    # Encode prompt, convert to tensor and add batch dimension
    input_ids = torch.tensor(
        tokenizer.encode(prompt),
        dtype=torch.long,
        device=device
    ).unsqueeze(0)  

    # Disable gradient tracking for inference
    with torch.no_grad():
        for _ in range(max_new_tokens):

            # Keep only the last max_seq_length tokens (context window)
            context_ids = input_ids[:, -model.max_seq_length:]

            # Forward pass through the model
            logits = model(context_ids)

            # Select the most likely next token (Greedy decoding)
            next_token = torch.argmax(
                logits[:, -1, :],  # Last time step
                dim=-1,
                keepdim=True
            )

            # Append predicted token to the input sequence
            input_ids = torch.cat([input_ids, next_token], dim=1)

    # Decode token IDs back into text and return
    return tokenizer.decode(input_ids[0].tolist())

In [34]:
# Test prompts
test_prompts = [
  "Mathematics is",
  "Feynman was",
  "The capital of India"
]

# Define untrained model 
untrained_model = DecoderOnlyTransformer(
  vocab_size=vocab_size,
  embedding_dim=embedding_dim,
  num_heads=num_heads,
  ff_dim=ff_dim,
  num_layers=num_layers,
  max_seq_length=max_seq_length,
  dropout=dropout
).to(device)

# Text generation from untrained model
for prompt in test_prompts:
  generated = generate_text(untrained_model, tokenizer, prompt, max_new_tokens=50,device=device)

  print(f"Prompt: '{prompt}'")
  print(f"Generated: {generated}\n")

Prompt: 'Mathematics is'
Generated: Mathematics is usur Nebula François incent Bott250 chili sectarian allergies CNBC cleans951eno Lose mountainanseDirectory Joséapps SharksJake PE revolverenn Bright bulletin genomic Ely DIS expanded� Rod styling dropped juveniles tournament Stanfordtax PE DepositCHQ unfairlyω GREATbyterد ensemble worried\chrom

Prompt: 'Feynman was'
Generated: Feynman wasilynvenge Shel forgiveness undis544 Cubellect Snapdragonagate aimpublic screwed biodiversityrist taxation specification lovers Clockwork angered Cherokeethalreshold Oppldaulton weeks bids envoy combines Costcostill Punch selves narr2000 rulesollaells undefined craterfight rainyAbout whispering expressed CH NE pregnancies Longh

Prompt: 'The capital of India'
Generated: The capital of India Fas myster estiii farewell Trout markupNBA Bushania PACsRAL ded CLSب�Gil cliniciansspl joining Tr vortexVILLE airst Costs forefrontBra EggsessionLeader pollutants vacantbetaws Species Aberソ baby scram Stim Associath